# 01. 인프라 배포 및 AI Search 파이프라인 설정

이 노트북에서는 Bicep으로 전체 인프라를 배포하고, Azure AI Search 파이프라인(인덱서 + 스킬셋)을 설정합니다.

## 지원 리전
| 리전 | Bicep 경로 | 특이사항 |
|------|-----------|----------|
| **Sweden Central** | `infra/sweden/` | DI 동일 리전, 단일 RG |
| **Korea Central** | `infra/korea/` | DI East US 2 Cross-Region PE, 2개 RG |

## 배포 리소스
| 리소스 | SKU | 용도 |
|--------|-----|------|
| Virtual Network (10.0.0.0/16) | - | Private Network 격리 |
| Storage Account | Standard LRS | 크롤링 데이터 저장 (Private) |
| Azure AI Services | S0 | GPT-5.4, text-embedding-3-large |
| Document Intelligence | S0 | PDF 레이아웃 분석 |
| Azure AI Search | **Standard (S1)** | 벡터 + 시맨틱 하이브리드 검색 |
| Function App (EP1) | Elastic Premium | Python 크롤러 (VNet Integration) |
| Logic App | Consumption | 크롤 스케줄러 (매일 21:00 UTC) |
| Private Endpoints × 4+ | - | 모든 서비스 프라이빗 접근 |
| Shared Private Links × 3 | - | AI Search 아웃바운드 |

## 노트북 흐름
```
1. 리전 선택 + 사전 요구사항 확인
2. Bicep 배포 (선택 리전, Private Network)
3. Shared Private Link 배포 + 승인
4. Function 코드 배포 (crawl + preprocess + custom skills 3개)
5. AI Search 파이프라인 설정 (Index + Skillset + Indexer)
6. .env 파일 생성
7. 배포 검증
```

## 1. 리전 선택 + 사전 요구사항 확인

아래 셀에서 배포할 리전을 선택합니다. `DEPLOY_REGION`을 `"sweden"` 또는 `"korea"`로 설정하세요.

In [1]:
# ╔════════════════════════════════════════════════════════════════════╗
# ║  배포 옵션 선택                                                    ║
# ║   - "sweden"        : Sweden Central, Private Network (PE + JumpVM) ║
# ║   - "korea"         : Korea Central,  Private Network (PE + JumpVM) ║
# ║   - "sweden-public" : Sweden Central, ALL public (workshop, no PE)  ║
# ╚════════════════════════════════════════════════════════════════════╝
DEPLOY_REGION = "sweden-public"   # ← 변경: "sweden" / "korea" / "sweden-public"

assert DEPLOY_REGION in ("sweden", "korea", "sweden-public"), f"Invalid region: {DEPLOY_REGION}"
print(f"✅ 배포 옵션: {DEPLOY_REGION}")

✅ 배포 옵션: sweden-public


In [2]:
# Azure CLI 로그인 확인
!az account show --output table

EnvironmentName    HomeTenantId                          IsDefault    Name                            State    TenantDefaultDomain               TenantDisplayName    TenantId
-----------------  ------------------------------------  -----------  ------------------------------  -------  --------------------------------  -------------------  ------------------------------------
AzureCloud         ebed7cce-48ed-45fe-ad61-fddfc8e2fb54  True         ME-MngEnvMCAP719772-jihyeseo-1  Enabled  MngEnvMCAP719772.onmicrosoft.com  Contoso              ebed7cce-48ed-45fe-ad61-fddfc8e2fb54


In [3]:
import subprocess
import json
from getpass import getpass

# 구독 ID
result = subprocess.run(
    ["az", "account", "show", "--query", "id", "--output", "tsv"],
    capture_output=True, text=True
)
SUBSCRIPTION_ID = result.stdout.strip()

# ── 리전별 설정 ──
# ※ 양쪽 Bicep 모두 GPT 파라미터명: gptDeploymentName / gptModelVersion
REGION_CONFIG = {
    "sweden": {
        "location": "swedencentral",
        "rg_name": "rg-rag-indexing-lab-swc",
        "deployment_name": "rag-indexing-lab-swc",
        "bicep_path": "../infra/sweden/main.bicep",
        "spl_bicep_path": "../infra/sweden/modules/ai-search-spl.bicep",
        "has_di_rg": False,
        "is_public": False,
    },
    "korea": {
        "location": "koreacentral",
        "rg_name": "rg-rag-indexing-lab-krc",
        "deployment_name": "rag-indexing-lab-krc",
        "bicep_path": "../infra/korea/main.bicep",
        "spl_bicep_path": "../infra/korea/modules/ai-search-spl.bicep",
        "has_di_rg": True,
        "di_rg_name": "rg-rag-indexing-lab-eus2",
        "is_public": False,
    },
    "sweden-public": {
        "location": "swedencentral",
        "rg_name": "rg-rag-indexing-lab-swc-pub",
        "deployment_name": "rag-indexing-lab-swc-pub",
        "bicep_path": "../infra/sweden-public/main.bicep",
        "spl_bicep_path": None,        # public 변형은 SPL 불필요
        "has_di_rg": False,
        "is_public": True,
    },
}

cfg = REGION_CONFIG[DEPLOY_REGION]
RG_NAME = cfg["rg_name"]
LOCATION = cfg["location"]
DEPLOYMENT_NAME = cfg["deployment_name"]
BICEP_PATH = cfg["bicep_path"]
IS_PUBLIC = cfg["is_public"]

# JumpVM 설정 — public 변형은 JumpVM 자체가 없음
if IS_PUBLIC:
    JUMPVM_ADMIN_PASSWORD = None
    JUMPVM_ENTRA_USER_OBJECT_IDS = None
    print("ℹ️  public 변형: JumpVM/PE 없음 — 비밀번호 입력 생략")
else:
    JUMPVM_ADMIN_PASSWORD = getpass("JumpVM admin password: ")
    JUMPVM_ENTRA_USER_OBJECT_IDS = "['4cb458b5-ef0c-403d-8162-c39855018986']"

print(f"Region:      {DEPLOY_REGION} ({LOCATION})")
print(f"RG:          {RG_NAME}")
print(f"Bicep:       {BICEP_PATH}")
print(f"Deployment:  {DEPLOYMENT_NAME}")
print(f"Network:     {'PUBLIC (no PE)' if IS_PUBLIC else 'PRIVATE (PE + JumpVM)'}")
print(f"Subscription: {SUBSCRIPTION_ID}")

ℹ️  public 변형: JumpVM/PE 없음 — 비밀번호 입력 생략
Region:      sweden-public (swedencentral)
RG:          rg-rag-indexing-lab-swc-pub
Bicep:       ../infra/sweden-public/main.bicep
Deployment:  rag-indexing-lab-swc-pub
Network:     PUBLIC (no PE)
Subscription: 8fcfff7e-0a34-491c-a1e5-23b22c1736de


In [4]:
# 필수 도구 확인
!az bicep version
!func --version  # Azure Functions Core Tools

==Bicep CLI version 0.43.8 (310735909d)

=4.10.0


## 2. Bicep 배포

Private Network(VNet + Private Endpoints) 전체 인프라를 한 번에 배포합니다.  
약 10~15분 소요됩니다.

- **Sweden Central**: 단일 RG, DI 동일 리전
- **Korea Central**: 메인 RG(Korea Central) + DI RG(East US 2), Cross-Region PE

In [6]:
# 리전별 Bicep 배포 (~10분 소요)
# ※ 모든 Bicep 공통 파라미터명: gptDeploymentName / gptModelVersion
import re
import time

# 현재 로그인 사용자 OID 추출 → 데이터 평면 RBAC 자동 부여
# (AI Search: Search Service Contributor + Search Index Data Contributor 등)
_oid_res = subprocess.run(
    ["az", "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"],
    capture_output=True, text=True
)
CURRENT_USER_OID = _oid_res.stdout.strip()
assert CURRENT_USER_OID, f"signed-in-user OID 조회 실패: {_oid_res.stderr}"
USER_OBJECT_IDS_JSON = json.dumps([CURRENT_USER_OID])
print(f"current user OID: {CURRENT_USER_OID}")

deploy_cmd = [
    "az", "deployment", "sub", "create",
    "--location", LOCATION,
    "--template-file", BICEP_PATH,
    "--name", DEPLOYMENT_NAME,
    "--parameters", f"resourceGroupName={RG_NAME}",
    "--parameters", f"location={LOCATION}",
    "--parameters", "embeddingDeploymentName=text-embedding-3-large",
    "--parameters", "gptDeploymentName=gpt-5.4",
    "--parameters", "gptModelVersion=2026-03-05",
    "--parameters", "searchSku=standard",
    "--parameters", "blobContainerName=raw-documents",
    "--parameters", f"userObjectIds={USER_OBJECT_IDS_JSON}",
]
# JumpVM 파라미터는 private 변형에만 추가 (sweden-public 은 JumpVM 모듈 자체가 없음)
if not IS_PUBLIC:
    deploy_cmd += [
        "--parameters", "jumpvmAdminUsername=azureadmin",
        "--parameters", f"jumpvmAdminPassword={JUMPVM_ADMIN_PASSWORD}",
        "--parameters", f"jumpvmEntraUserObjectIds={JUMPVM_ENTRA_USER_OBJECT_IDS}",
    ]
deploy_cmd += ["--output", "table"]

max_attempts = 4
result = None

for attempt in range(1, max_attempts + 1):
    print(f"배포 시작: {DEPLOY_REGION} ({LOCATION}) [attempt {attempt}/{max_attempts}]")
    print(f"Bicep: {BICEP_PATH}")
    result = subprocess.run(deploy_cmd, capture_output=True, text=True)
    print(result.stdout)

    if result.returncode == 0:
        print("✓ 배포 성공")
        break

    stderr = result.stderr or ""
    print("ERROR:", stderr)

    # 1) Soft-delete 충돌 자동 복구: 계정 purge 후 재시도
    if "FlagMustBeSetForRestore" in stderr and "Microsoft.CognitiveServices/accounts/" in stderr:
        name_match = re.search(r"Microsoft\.CognitiveServices/accounts/([^'\\s]+)", stderr)
        rg_match = re.search(r"/resourceGroups/([^/]+)/providers/Microsoft\.CognitiveServices/accounts/", stderr)

        soft_deleted_name = name_match.group(1) if name_match else None
        purge_rg = rg_match.group(1) if rg_match else RG_NAME

        if soft_deleted_name:
            print(f"\n⚠️  Soft-deleted Cognitive 계정 감지: {soft_deleted_name} (RG: {purge_rg})")
            purge_cmd = [
                "az", "cognitiveservices", "account", "purge",
                "--name", soft_deleted_name,
                "--location", LOCATION,
                "--resource-group", purge_rg,
            ]
            purge_result = subprocess.run(purge_cmd, capture_output=True, text=True)

            if purge_result.returncode == 0:
                print("✓ purge 완료. 20초 대기 후 배포 재시도")
                time.sleep(20)
                continue
            else:
                print("ERROR (purge):", purge_result.stderr)
                break
        else:
            print("Soft-delete 계정명을 stderr에서 파싱하지 못했습니다. 수동 purge가 필요합니다.")
            break

    # 2) Key Vault soft-delete 충돌 자동 복구: vault purge 후 재시도
    if "vault with the same name already exists in deleted state" in stderr.lower():
        print("\n⚠️  Key Vault soft-delete 감지 — vault 이름 파싱 중...")
        
        # 패턴 1: 리소스 경로에서 추출 (Microsoft.KeyVault/vaults/...)
        vault_match = re.search(r"Microsoft\.KeyVault/vaults/([a-z0-9\-]+)", stderr, re.IGNORECASE)
        vault_name = vault_match.group(1) if vault_match else None
        
        # 패턴 2: 일반적인 Key Vault 이름 형식 (kv-ragi-<suffix>)
        if not vault_name:
            # suffix 기반으로 구성 시도 (STORAGE_ACCOUNT 또는 outputs에서 추출)
            try:
                storage_suffix = STORAGE_ACCOUNT.replace('stragi', '') if 'STORAGE_ACCOUNT' in dir() else ""
                if not storage_suffix:
                    storage_suffix = outputs.get('storageAccountName', {}).get('value', '').replace('stragi', '')
                vault_name = f"kv-ragi-{storage_suffix}" if storage_suffix else None
            except:
                vault_name = None
        
        # 패턴 3: stderr에서 키 워드 주변 텍스트 추출
        if not vault_name:
            vault_match = re.search(r"['\"`]([a-z0-9\-]+)['\"`]?\s+(?:already exists|with the same name)", stderr, re.IGNORECASE)
            vault_name = vault_match.group(1) if vault_match else None
        
        # 디버그: 파싱된 vault 이름 출력
        print(f"  파싱된 Key Vault 이름: {vault_name if vault_name else '(찾을 수 없음)'}")
        
        if vault_name:
            print(f"⚙️  Key Vault purge 시작: {vault_name} (location: {LOCATION})")
            purge_cmd = [
                "az", "keyvault", "purge",
                "--name", vault_name,
                "--location", LOCATION,
            ]
            purge_result = subprocess.run(purge_cmd, capture_output=True, text=True)
            
            if purge_result.returncode == 0:
                print("✓ Key Vault purge 완료. 20초 대기 후 배포 재시도")
                time.sleep(20)
                continue
            else:
                print("⚠️  Key Vault purge 실패:")
                print("  stdout:", purge_result.stdout[:300] if purge_result.stdout else "(empty)")
                print("  stderr:", purge_result.stderr[:300] if purge_result.stderr else "(empty)")
                # purge 실패해도 한 번 더 시도 가능
                if attempt < max_attempts:
                    print("재시도 중...")
                    time.sleep(10)
                    continue
                break
        else:
            print("\n❌ Key Vault 이름을 파싱하지 못했습니다.")
            print("\n=== 수동 purge 필요 ===")
            print("다음 명령으로 soft-deleted vault를 찾으세요:")
            print(f"  az keyvault list-deleted --resource-type vault --location {LOCATION} --query \"[].name\" -o tsv")
            print("\n아래 명령으로 vault를 purge하세요:")
            print(f"  az keyvault purge --name <vault-name> --location {LOCATION}")
            print("\n그 후 이 셀을 다시 실행하세요.")
            break

    # 3) Storage 계정 배타 작업 중이면 잠시 대기 후 재시도
    if "StorageAccountOperationInProgress" in stderr:
        if attempt < max_attempts:
            print("⚠️  Storage 작업 진행 중. 30초 대기 후 재시도")
            time.sleep(30)
            continue
        print("Storage 계정 작업 충돌이 지속되어 재시도 한도를 초과했습니다.")
        break

    # 그 외 에러는 즉시 중단
    break

if result is not None and result.returncode != 0:
    print("배포 실패: 위 ERROR 로그를 확인하세요.")

current user OID: 4cb458b5-ef0c-403d-8162-c39855018986
배포 시작: sweden-public (swedencentral) [attempt 1/4]
Bicep: ../infra/sweden-public/main.bicep

ERROR: WARNING: /home/azureuser/localfiles/azure-ai-search-deepdive/infra/sweden-public/modules/foundry-hub.bicep(18,7) : Warning no-unused-params: Parameter "storageAccountName" is declared but never used. [https://aka.ms/bicep/linter-diagnostics#no-unused-params]
/home/azureuser/localfiles/azure-ai-search-deepdive/infra/sweden-public/modules/foundry-hub.bicep(24,7) : Warning no-unused-params: Parameter "searchServiceName" is declared but never used. [https://aka.ms/bicep/linter-diagnostics#no-unused-params]
/home/azureuser/localfiles/azure-ai-search-deepdive/infra/sweden-public/modules/foundry-hub.bicep(33,7) : Warning no-unused-params: Parameter "aiServicesName" is declared but never used. [https://aka.ms/bicep/linter-diagnostics#no-unused-params]

ERROR: {"status":"Failed","error":{"code":"DeploymentFailed","target":"/subscriptions/8fcf

In [8]:
# 배포 출력값 파싱
result = subprocess.run(
    ["az", "deployment", "sub", "show",
     "--name", DEPLOYMENT_NAME,
     "--query", "properties.outputs",
     "--output", "json"],
    capture_output=True, text=True
)
outputs = json.loads(result.stdout)

print(f"=== 배포된 리소스 ({DEPLOY_REGION}) ===")
for key, val in outputs.items():
    print(f"  {key}: {val['value']}")

# GPT deployment name (양쪽 Bicep 동일: gptDeploymentName)
GPT_DEPLOY_NAME = outputs.get('gptDeploymentName', {}).get('value', 'gpt-5.4')

# 모든 핵심 출력값 추출
SEARCH_NAME = outputs['aiSearchName']['value']
STORAGE_NAME = outputs['storageAccountName']['value']
AI_SERVICES_NAME = outputs['openaiAccountName']['value']

# suffix 추출 (storageAccountName = 'stragi<suffix>')
_suffix = STORAGE_NAME.replace('stragi', '')
DI_NAME = f"di-ragi-{_suffix}"

# DI RG — Sweden: 메인 RG, Korea: 별도 RG (East US 2)
DI_RG = cfg.get("di_rg_name", RG_NAME) if cfg["has_di_rg"] else RG_NAME

# STORAGE_ACCOUNT는 STORAGE_NAME과 동일
STORAGE_ACCOUNT = STORAGE_NAME

# 리소스 ID 직접 구성 (Azure CLI 대신)
STORAGE_RESOURCE_ID = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RG_NAME}/providers/Microsoft.Storage/storageAccounts/{STORAGE_ACCOUNT}"

if IS_PUBLIC:
    print("ℹ️  public 변형: SPL/PE 배포 건너뜀 (모든 리소스 public network)")
else:
    # private 변형: DI, AI Services 리소스 ID도 구성
    AI_SERVICES_RESOURCE_ID = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RG_NAME}/providers/Microsoft.CognitiveServices/accounts/{AI_SERVICES_NAME}"
    DI_RESOURCE_ID = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{DI_RG}/providers/Microsoft.CognitiveServices/accounts/{DI_NAME}"

print(f"✓ 주요 변수 추출 완료: STORAGE_ACCOUNT={STORAGE_ACCOUNT}, SEARCH_NAME={SEARCH_NAME}")
print(f"✓ STORAGE_RESOURCE_ID={STORAGE_RESOURCE_ID}")

=== 배포된 리소스 (sweden-public) ===
  aiSearchEndpoint: https://search-ragi-63325wdo.search.windows.net
  aiSearchName: search-ragi-63325wdo
  aiSearchPrincipalId: 79f4ea79-4b31-40a7-bf09-47178a892899
  crawlConsFunctionName: func-crawl-cons-ragi-63325wdo
  crawlFunctionName: func-crawl-cons-ragi-63325wdo
  crawlFunctionUrl: https://func-crawl-cons-ragi-63325wdo.azurewebsites.net/api/orchestrators/crawl_preprocess
  crawlLogicAppName: logic-crawl-ragi-63325wdo
  docIntelligenceEndpoint: https://di-ragi-63325wdo.cognitiveservices.azure.com/
  foundryHubName: hub-ragi-63325wdo
  foundryKeyVaultName: kv-ragi-63325wdo
  foundryProjectEndpoint: https://ais-ragi-63325wdo.services.ai.azure.com/api/projects/proj-ragi-default
  foundryProjectName: proj-ragi-63325wdo
  gptDeploymentName: gpt-5.4
  location: swedencentral
  openaiAccountName: ais-ragi-63325wdo
  openaiEndpoint: https://ais-ragi-63325wdo.cognitiveservices.azure.com/
  preprocessFunctionName: func-preprocess-ragi-63325wdo
  preprocessF

## 3. Shared Private Link 배포 + 승인

### 3a. SPL 배포
AI Search가 아웃바운드로 Storage, AI Services, Document Intelligence에 접근하기 위한  
Shared Private Link를 배포합니다. **초회 1회만 실행**합니다.

> SPL이 Approved 상태가 되면 재배포 시 ARM이 "conflicting update" 오류를 발생시키므로 메인 Bicep과 분리

In [9]:
# 핵심 변수 확인 (모두 cell 9에서 추출됨)
print("STORAGE_ACCOUNT:", STORAGE_ACCOUNT)
print("RG_NAME:", RG_NAME)
print("STORAGE_RESOURCE_ID:", STORAGE_RESOURCE_ID)
print("SEARCH_NAME:", SEARCH_NAME)
print("✓ 모든 필수 변수가 정의됨")

STORAGE_ACCOUNT: stragi63325wdoby
RG_NAME: rg-rag-indexing-lab-swc-pub
STORAGE_RESOURCE_ID: /subscriptions/8fcfff7e-0a34-491c-a1e5-23b22c1736de/resourceGroups/rg-rag-indexing-lab-swc-pub/providers/Microsoft.Storage/storageAccounts/stragi63325wdoby
SEARCH_NAME: search-ragi-63325wdo
✓ 모든 필수 변수가 정의됨


In [10]:
SEARCH_NAME = outputs['aiSearchName']['value']
STORAGE_NAME = outputs['storageAccountName']['value']
AI_SERVICES_NAME = outputs['openaiAccountName']['value']

# suffix 추출 (storageAccountName = 'stragi<suffix>')
_suffix = STORAGE_NAME.replace('stragi', '')
DI_NAME = f"di-ragi-{_suffix}"

# DI RG — Sweden: 메인 RG, Korea: 별도 RG (East US 2)
DI_RG = cfg.get("di_rg_name", RG_NAME) if cfg["has_di_rg"] else RG_NAME

if IS_PUBLIC:
    print("ℹ️  public 변형: SPL/PE 배포 건너뜀 (모든 리소스 public network)")
else:
    AI_SERVICES_RESOURCE_ID = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RG_NAME}/providers/Microsoft.CognitiveServices/accounts/{AI_SERVICES_NAME}"
    DI_RESOURCE_ID = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{DI_RG}/providers/Microsoft.CognitiveServices/accounts/{DI_NAME}"

    # SPL Bicep 배포 (초회 1회만)
    spl_bicep = cfg["spl_bicep_path"]
    print(f"SPL 배포: {spl_bicep}")
    print(f"  → spl-blob:        {STORAGE_NAME}")
    print(f"  → spl-aiservices:  {AI_SERVICES_NAME}")
    print(f"  → spl-docintel:    {DI_NAME} (RG: {DI_RG})")

    spl_result = subprocess.run([
        "az", "deployment", "group", "create",
        "--resource-group", RG_NAME,
        "--template-file", spl_bicep,
        "--parameters", f"searchServiceName={SEARCH_NAME}",
        "--parameters", f"storageAccountId={STORAGE_RESOURCE_ID}",
        "--parameters", f"aiServicesId={AI_SERVICES_RESOURCE_ID}",
        "--parameters", f"docIntelligenceId={DI_RESOURCE_ID}",
        "--output", "table",
    ], capture_output=True, text=True)
    print(spl_result.stdout)
    if spl_result.returncode != 0:
        print("ERROR (이미 배포됨이면 정상):", spl_result.stderr[:500])

ℹ️  public 변형: SPL/PE 배포 건너뜀 (모든 리소스 public network)


### 3b. SPL 승인

SPL 배포 후 약 5~10분 대기 후, 각 리소스에서 Pending 연결 요청을 자동 승인합니다.

In [11]:
# SPL 상태 확인 (public 변형은 SPL 자체가 없음)
if IS_PUBLIC:
    print("ℹ️  public 변형: SPL 없음 — 건너뜀")
else:
    print(f"AI Search: {SEARCH_NAME}")
    !az search shared-private-link-resource list \
        --service-name {SEARCH_NAME} \
        --resource-group {RG_NAME} \
        --query "[].{{name:name, status:properties.status}}" \
        --output table

ℹ️  public 변형: SPL 없음 — 건너뜀


In [12]:
# Storage Account SPL 승인
if IS_PUBLIC:
    print("ℹ️  public 변형: PE 승인 불필요 — 건너뜀")
    pending = []
else:
    result = subprocess.run(
        ["az", "network", "private-endpoint-connection", "list",
         "--name", STORAGE_NAME,
         "--resource-group", RG_NAME,
         "--type", "Microsoft.Storage/storageAccounts",
         "--query", "[?properties.privateLinkServiceConnectionState.status=='Pending'].name",
         "--output", "tsv"],
        capture_output=True, text=True
    )
    pending = [c for c in result.stdout.strip().split('\n') if c]

for conn in pending:
    print(f"승인: {conn}")
    subprocess.run(
        ["az", "network", "private-endpoint-connection", "approve",
         "--name", conn,
         "--resource-name", STORAGE_NAME,
         "--resource-group", RG_NAME,
         "--type", "Microsoft.Storage/storageAccounts",
         "--description", "Approved for AI Search indexer",
         "--output", "none"],
        check=False,
    )

if not pending:
    print("승인 대기 중인 Storage 연결 없음 (이미 승인됨 또는 아직 생성 중)")

ℹ️  public 변형: PE 승인 불필요 — 건너뜀
승인 대기 중인 Storage 연결 없음 (이미 승인됨 또는 아직 생성 중)


In [13]:
# AI Services + Document Intelligence SPL 승인
# Korea: DI는 별도 RG에 있으므로 리소스별 RG 매핑
if IS_PUBLIC:
    print("ℹ️  public 변형: PE 승인 불필요 — 건너뜀")
    spl_targets = []
else:
    spl_targets = [
        (AI_SERVICES_NAME, RG_NAME),
        (DI_NAME, DI_RG),
    ]

approved_count = 0
for resource_name, rg in spl_targets:
    result = subprocess.run(
        ["az", "network", "private-endpoint-connection", "list",
         "--name", resource_name,
         "--resource-group", rg,
         "--type", "Microsoft.CognitiveServices/accounts",
         "--query", "[?properties.privateLinkServiceConnectionState.status=='Pending'].name",
         "--output", "tsv"],
        capture_output=True, text=True
    )

    pending = [c for c in result.stdout.strip().split('\n') if c]
    if not pending:
        print(f"대기 연결 없음: {resource_name} (RG: {rg})")
        continue

    for conn in pending:
        print(f"승인: {resource_name} / {conn} (RG: {rg})")
        subprocess.run(
            ["az", "network", "private-endpoint-connection", "approve",
             "--name", conn,
             "--resource-name", resource_name,
             "--resource-group", rg,
             "--type", "Microsoft.CognitiveServices/accounts",
             "--description", "Approved for AI Search skillset",
             "--output", "none"],
            check=False,
        )
        approved_count += 1

if not IS_PUBLIC:
    print(f"승인 처리 건수: {approved_count}")

ℹ️  public 변형: PE 승인 불필요 — 건너뜀


In [14]:
# 최종 승인 상태 확인 — 모두 Approved 여야 AI Search 파이프라인 작동
if IS_PUBLIC:
    print("ℹ️  public 변형: SPL 없음 — 건너뜀")
else:
    !az search shared-private-link-resource list \
        --service-name {SEARCH_NAME} \
        --resource-group {RG_NAME} \
        --query "[].{{name:name, status:properties.status, groupId:properties.groupId}}" \
        --output table

ℹ️  public 변형: SPL 없음 — 건너뜀


## 4. Function App 코드 배포

Bicep으로 Function App 인프라만 생성했으므로, 아래 3개 Function 기반 구성요소 코드를 순차 배포합니다.

- Crawl pipeline 수집 함수: `logic-apps/crawl-function`
- Crawl pipeline 전처리 함수: `logic-apps/preprocess-function`
- Indexing custom skills (3개): `skills-function` (`markdown_split`, `pptx_page_split`, `verbalize`)

In [15]:
import subprocess, tempfile, time, zipfile
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

# ── 3개 Function App 매핑 (모두 Flex Consumption FC1 SKU) ──
#  · func-crawl-cons-…      : Durable Functions orchestrator (logic-apps/crawl-function)
#  · func-preprocess-…      : 수집 후 전처리 (logic-apps/preprocess-function)
#  · func-skills-…          : AI Search Custom Web API Skills 3종 (skills-function)
_suffix = outputs["storageAccountName"]["value"].replace("stragi", "")
CRAWL_CONS_FUNC_APP = (
    outputs.get("crawlConsFunctionName", {}).get("value")
    or f"func-crawl-cons-ragi-{_suffix[:8]}"
)
PREPROCESS_FUNC_APP = (
    outputs.get("preprocessFunctionName", {}).get("value")
    or f"func-preprocess-ragi-{_suffix}"
)
SKILLS_FUNC_APP = (
    outputs.get("skillsFunctionName", {}).get("value")
    or f"func-skills-ragi-{_suffix}"
)

# crawlFunctionName 은 이제 FC1 crawl-cons 를 가리킴
CRAWL_FUNC_APP = outputs.get("crawlFunctionName", {}).get("value", CRAWL_CONS_FUNC_APP)
FUNC_APP_NAME = CRAWL_FUNC_APP  # 하위 셀 호환

# custom skills 앱은 GPT/DI endpoint 환경변수 필요
OPENAI_ENDPOINT = outputs["openaiEndpoint"]["value"]
DI_ENDPOINT     = outputs["docIntelligenceEndpoint"]["value"]
SKILLS_APP_SETTINGS = [
    f"AZURE_OPENAI_ENDPOINT={OPENAI_ENDPOINT}",
    f"AZURE_OPENAI_GPT54_DEPLOYMENT={GPT_DEPLOY_NAME}",
    f"AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT={DI_ENDPOINT}",
]

deploy_targets = [
    ("crawl-cons", CRAWL_CONS_FUNC_APP, repo_root / "logic-apps" / "crawl-function",      ["orchestrators/crawl_preprocess"]),
    ("preprocess", PREPROCESS_FUNC_APP, repo_root / "logic-apps" / "preprocess-function", ["preprocess?source=__ping"]),
    ("skills",     SKILLS_FUNC_APP,     repo_root / "skills-function",                    ["markdown_split", "pptx_page_split", "verbalize"]),
]


def _zip_dir_for_deploy(src_dir: Path) -> Path:
    """원격 빌드용 zip 생성 (의존성/캐시 제외 — requirements.txt 만 포함 → Oryx 가 원격에서 설치)."""
    EXCLUDE_DIRS = {"__pycache__", ".venv", ".python_packages", ".git", "crawl"}
    EXCLUDE_FILES = {"local.settings.json"}
    EXCLUDE_SUFFIX = (".pyc",)
    tmp = Path(tempfile.mkstemp(prefix=f"funczip-{src_dir.name}-", suffix=".zip")[1])
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in src_dir.rglob("*"):
            rel = path.relative_to(src_dir)
            if any(part in EXCLUDE_DIRS for part in rel.parts):
                continue
            if path.is_file():
                if path.name in EXCLUDE_FILES or path.suffix in EXCLUDE_SUFFIX:
                    continue
                zf.write(path, rel.as_posix())
    return tmp


def _verify_function(app_name: str, probe_paths: list[str], wait_s: int = 45) -> None:
    """Restart 후 cold-start 대기 → HTTP 헬스체크.
    Flex Consumption 은 cold start 가 길어 45s 정도 필요.
    """
    import requests as _r
    subprocess.run(
        ["az", "functionapp", "restart", "--name", app_name, "--resource-group", RG_NAME, "-o", "none"],
        check=False,
    )
    print(f"  → restart 후 {wait_s}s 대기 (Flex cold start)")
    time.sleep(wait_s)
    for p in probe_paths:
        url = f"https://{app_name}.azurewebsites.net/api/{p}"
        try:
            r = _r.get(url, timeout=60)
            mark = "✅" if r.status_code < 500 else "❌"
            print(f"  {mark} HTTP {r.status_code}  /api/{p}")
        except Exception as e:
            print(f"  ⚠️  /api/{p} — {e}")


for label, app_name, src_dir, probe_paths in deploy_targets:
    print(f"\n=== Deploy {label} → {app_name} ===")
    print(f"Source Dir: {src_dir}")
    if not src_dir.exists():
        raise FileNotFoundError(f"{label} 경로 없음: {src_dir}")

    # skills 앱은 publish 전에 app settings 먼저 적용
    if label == "skills":
        print(f"  → app settings ({len(SKILLS_APP_SETTINGS)} entries)")
        subprocess.run(
            ["az", "functionapp", "config", "appsettings", "set",
             "--name", app_name, "--resource-group", RG_NAME,
             "--settings", *SKILLS_APP_SETTINGS, "-o", "none"],
            check=False,
        )
        time.sleep(15)  # appsettings 변경 직후 deploy 충돌 회피

    # zip 생성 + az config-zip + --build-remote true (Flex 가 원격에서 requirements.txt 설치)
    # NOTE: Flex Consumption SKU 는 SCM_DO_BUILD_DURING_DEPLOYMENT / ENABLE_ORYX_BUILD 를 지원하지 않으므로
    #       해당 setting 을 추가하면 InvalidAppSettingsException 으로 배포가 거부됨. 절대 설정하지 말 것.
    zip_path = _zip_dir_for_deploy(src_dir)
    try:
        print(f"  → az zip deploy ({zip_path.stat().st_size // 1024} KB, build-remote)")
        result = subprocess.run(
            ["az", "functionapp", "deployment", "source", "config-zip",
             "--name", app_name, "--resource-group", RG_NAME,
             "--src", str(zip_path),
             "--build-remote", "true",
             "-o", "none"],
            capture_output=True, text=True, timeout=900,
        )
        if result.returncode != 0:
            print(f"ERROR ({label}):", (result.stderr or "")[-1500:])
            continue
        print(f"  ✓ {label} 배포 완료")
    finally:
        zip_path.unlink(missing_ok=True)

    # restart + HTTP 검증
    _verify_function(app_name, probe_paths)


=== Deploy crawl-cons → func-crawl-cons-ragi-63325wdo ===
Source Dir: /home/azureuser/localfiles/azure-ai-search-deepdive/logic-apps/crawl-function
  → az zip deploy (37 KB, build-remote)


  ✓ crawl-cons 배포 완료
  → restart 후 45s 대기 (Flex cold start)
  ✅ HTTP 202  /api/orchestrators/crawl_preprocess

=== Deploy preprocess → func-preprocess-ragi-63325wdo ===
Source Dir: /home/azureuser/localfiles/azure-ai-search-deepdive/logic-apps/preprocess-function
  → az zip deploy (5 KB, build-remote)
  ✓ preprocess 배포 완료
  → restart 후 45s 대기 (Flex cold start)
  ⚠️  /api/preprocess?source=__ping — HTTPSConnectionPool(host='func-preprocess-ragi-63325wdo.azurewebsites.net', port=443): Read timed out. (read timeout=60)

=== Deploy skills → func-skills-ragi-63325wdo ===
Source Dir: /home/azureuser/localfiles/azure-ai-search-deepdive/skills-function
  → app settings (3 entries)
  → az zip deploy (6 KB, build-remote)
  ✓ skills 배포 완료
  → restart 후 45s 대기 (Flex cold start)
  ⚠️  /api/markdown_split — HTTPSConnectionPool(host='func-skills-ragi-63325wdo.azurewebsites.net', port=443): Read timed out. (read timeout=60)
  ✅ HTTP 404  /api/pptx_page_split
  ✅ HTTP 404  /api/verbalize


In [16]:
import requests as req

# 상위 셀 미실행 대비 fallback
_suffix = outputs["storageAccountName"]["value"].replace("stragi", "")
CRAWL_CONS_FUNC_APP = globals().get(
    "CRAWL_CONS_FUNC_APP",
    outputs.get("crawlConsFunctionName", {}).get("value") or f"func-crawl-cons-ragi-{_suffix[:8]}",
)
PREPROCESS_FUNC_APP = globals().get(
    "PREPROCESS_FUNC_APP",
    outputs.get("preprocessFunctionName", {}).get("value") or f"func-preprocess-ragi-{_suffix}",
)
SKILLS_FUNC_APP = globals().get(
    "SKILLS_FUNC_APP",
    outputs.get("skillsFunctionName", {}).get("value") or f"func-skills-ragi-{_suffix}",
)

# crawlFunctionName 은 이제 FC1 crawl-cons 를 가리킴
CRAWL_FUNC_APP = globals().get("CRAWL_FUNC_APP", outputs.get("crawlFunctionName", {}).get("value", CRAWL_CONS_FUNC_APP))

check_targets = [
    ("crawl-cons", CRAWL_CONS_FUNC_APP, ["orchestrators/crawl_preprocess"]),
    ("preprocess", PREPROCESS_FUNC_APP, ["preprocess"]),
    ("skills", SKILLS_FUNC_APP, ["markdown_split", "pptx_page_split", "verbalize"]),
]

for label, app_name, probe_paths in check_targets:
    print(f"\n=== Function App 확인: {label} ({app_name}) ===")

    result = subprocess.run(
        ["az", "functionapp", "function", "list",
         "--name", app_name,
         "--resource-group", RG_NAME,
         "--output", "json"],
        capture_output=True, text=True
    )

    funcs = json.loads(result.stdout) if result.stdout.strip() else []
    if funcs:
        for f in funcs:
            short_name = f["name"].split("/")[-1]
            print(f"  ✅ {short_name}  →  {f.get('invokeUrlTemplate', '')}")
        continue

    print("  CLI 조회 결과 없음 (cold start 전 정상 가능). HTTP 엔드포인트 연결 확인 중...")
    for path in probe_paths:
        url = f"https://{app_name}.azurewebsites.net/api/{path}"
        try:
            r = req.get(url, timeout=30)
            state = "정상" if r.status_code < 500 else "오류"
            print(f"  HTTP {r.status_code} [{path}] — {state}")
        except Exception as e:
            print(f"  연결 실패 [{path}]: {e}")


=== Function App 확인: crawl-cons (func-crawl-cons-ragi-63325wdo) ===


  ✅ activity_crawl_detail_batch  →  None
  ✅ activity_list_seqs  →  None
  ✅ activity_preprocess_source  →  None
  ✅ crawl  →  https://func-crawl-cons-ragi-63325wdo.azurewebsites.net/api/crawl
  ✅ crawl_preprocess_orchestrator  →  None
  ✅ http_start_crawl_preprocess  →  https://func-crawl-cons-ragi-63325wdo.azurewebsites.net/api/orchestrators/crawl_preprocess

=== Function App 확인: preprocess (func-preprocess-ragi-63325wdo) ===
  ✅ preprocess  →  https://func-preprocess-ragi-63325wdo.azurewebsites.net/api/preprocess

=== Function App 확인: skills (func-skills-ragi-63325wdo) ===
  ✅ markdown_split  →  https://func-skills-ragi-63325wdo.azurewebsites.net/api/markdown_split
  ✅ pptx_page_split  →  https://func-skills-ragi-63325wdo.azurewebsites.net/api/pptx_page_split
  ✅ verbalize  →  https://func-skills-ragi-63325wdo.azurewebsites.net/api/verbalize


## 5. AI Search 파이프라인 설정

AI Search 네이티브 Indexer + Skillset을 생성합니다:
- **Data Source**: `raw-documents` Blob 컨테이너 감시
- **Skillset**: SplitSkill → AzureOpenAIEmbeddingSkill
- **Index**: `law-documents-index` (벡터 3072D + 시맨틱)
- **Indexer**: 매일 06:00 UTC 자동 실행

> **Note**: AI Search가 Private Network 내에 있으므로, 임시로 Public Access를 허용합니다.

In [17]:
# AI Search 임시 Public Access 허용 (private 변형에서만 필요)
if IS_PUBLIC:
    print("ℹ️ public 변형: AI Search public access 변경 불필요 — 건너뜀")
else:
    !az search service update \
        --name {SEARCH_NAME} \
        --resource-group {RG_NAME} \
        --public-network-access enabled \
        --output none

    print("AI Search public access 임시 허용")

ℹ️ public 변형: AI Search public access 변경 불필요 — 건너뜀


### 5a. 법령 인덱싱 파이프라인 등록 (4개 인덱스)

크롤링된 한국 법률 데이터(JSON)를 대상으로 4개의 AI Search 인덱스 + 인덱서 + 스킬셋을 등록합니다.

| Source | Index | 대상 데이터 |
|--------|-------|-------------|
| `prec` | `prec-court-index` | 대법원 판례 |
| `detc` | `detc-court-index` | 헌법재판소 결정례 |
| `expc` | `expc-court-index` | 법제처 법령해석례 |
| `admrul` | `admrul-court-index` | 행정심판 재결례 |

각 인덱스는 **벡터(3072D) + 시맨틱 하이브리드 검색**과 **AzureOpenAIEmbeddingSkill** 스킬셋을 포함하며, 인덱서는 `PT24H`(24시간) 주기로 스케줄됩니다.

> **선행 조건**: 4단계 Function App 배포가 완료되어 raw JSON이 `raw-documents` 컨테이너에 존재해야 합니다.  
> 데이터가 없으면 인덱서는 0건 처리로 종료됩니다.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

SEARCH_ENDPOINT = outputs['aiSearchEndpoint']['value']
OPENAI_ENDPOINT = outputs['openaiEndpoint']['value']
OPENAI_ACCOUNT  = outputs['openaiAccountName']['value']
STORAGE_ACCOUNT = outputs['storageAccountName']['value']

# 환경변수 세팅 후 파이프라인 설정
os.environ.update({
    "AZURE_SEARCH_SERVICE_ENDPOINT": SEARCH_ENDPOINT,
    "AZURE_OPENAI_ENDPOINT": OPENAI_ENDPOINT,
    "AZURE_OPENAI_ACCOUNT_NAME": OPENAI_ACCOUNT,
    "AZURE_STORAGE_ACCOUNT_NAME": STORAGE_ACCOUNT,
    "AZURE_STORAGE_RESOURCE_ID": STORAGE_RESOURCE_ID,
    "AZURE_RESOURCE_GROUP": RG_NAME,
})

from src.pipeline import setup_legal_pipeline
setup_legal_pipeline(source="all")

AI Search 법률 인덱싱 파이프라인 설정
  Search    : https://search-ragi-63325wdo.search.windows.net
  Container : processed-documents
  스케줄    : PT24H (start: 2026-01-01T06:00:00Z)
  대상      : prec, detc, expc, admrul

[사전 확인] AI Search Shared Private Link 승인 필요:
  az search shared-private-link-resource list \
      --service-name <search-name> --resource-group <rg>
  → status가 'Approved'인지 확인


────────────────────────────────────────────────────────────
  [prec] prec-court-index
────────────────────────────────────────────────────────────
[Index] prec-court-index
  ✓ Index 'prec-court-index' 생성 완료
[Skillset] prec-rag-skillset
  ✓ Skillset 'prec-rag-skillset' 생성 완료 (임베딩: /document/판결요지)
[DataSource] prec-blob-datasource
  ✓ DataSource 'prec-blob-datasource' 생성 완료 (processed-documents/prec/)
[Indexer] prec-blob-indexer
  ✓ Indexer 'prec-blob-indexer' 생성 완료 (매 PT24H 실행)

────────────────────────────────────────────────────────────
  [detc] const-court-index
─────────────────────────────────────────

### 5b. 멀티모달 파이프라인 등록 (3종)

PDF/PPTX 문서를 위한 **세 가지** 멀티모달 인덱싱 파이프라인을 등록합니다 (각각 독립 인덱서):

| # | Pipeline | 대상 확장자 | 분할 방식 | Index 이름 |
|---|----------|------------|-----------|-----------|
| 1 | **PDF (basic)** | `.pdf` | DI Layout → `markdown_split` (header + length) → Embedding | `st-multimodal-pdf-index` |
| 2 | **PPTX (basic)** | `.pptx` | DI Layout → `pptx_page_split` (slide 단위) → Embedding | `st-multimodal-pptx-index` |
| 3 | **Verbalized** | `.pdf` | DI Layout → **GPT-5.4 이미지/도표 설명** → Markdown 분할 → Embedding | `st-multimodal-verbalized-index` |

3개 인덱서는 동일한 Data Source(`raw/`)를 공유하며 `indexedFileNameExtensions`로 자기 타입만 처리합니다.

> 선행 조건: `skills-function` (= `func-skills-ragi-…`) 배포 완료 — 4단계에서 처리됨.


In [19]:
# 4단계에서 skills-function 배포를 이미 수행했는지 확인
# NOTE: FC1 (Flex Consumption) 은 `az functionapp show --query state` 가 null 을 반환하므로
#       `az functionapp list` 로 상태를 조회해야 정확함
print("skills-function 배포는 4단계에서 수행됨")
print(f"Target Function App: {SKILLS_FUNC_APP}")

check = subprocess.run(
    ["az", "functionapp", "list",
     "--resource-group", RG_NAME,
     "--query", f"[?name=='{SKILLS_FUNC_APP}'].{{name:name, state:state}}",
     "--output", "json"],
    capture_output=True, text=True
)
apps = json.loads(check.stdout) if check.stdout.strip() else []
if apps:
    print(f"  state: {apps[0].get('state', '(unknown)')}")
else:
    print(f"WARN: Function App '{SKILLS_FUNC_APP}' 이 리소스 그룹에 없습니다.")

skills-function 배포는 4단계에서 수행됨
Target Function App: func-skills-ragi-63325wdo


  state: Running


In [20]:
# Skills Function Key 조회
import json, subprocess

key_result = subprocess.run(
    ["az", "functionapp", "keys", "list",
     "--name", SKILLS_FUNC_APP,
     "--resource-group", RG_NAME,
     "--query", "functionKeys.default",
     "--output", "tsv"],
    capture_output=True, text=True
)
SKILLS_FUNCTION_KEY = key_result.stdout.strip()
SKILLS_FUNCTION_URL = f"https://{SKILLS_FUNC_APP}.azurewebsites.net"

print(f"Skills Function URL: {SKILLS_FUNCTION_URL}")
print(f"Skills Function Key: {SKILLS_FUNCTION_KEY[:12]}..." if SKILLS_FUNCTION_KEY else "ERROR: Key not found")

Skills Function URL: https://func-skills-ragi-63325wdo.azurewebsites.net
Skills Function Key: 7hcRYIKTl2ia...


In [ ]:
# 멀티모달 파이프라인 등록 (PDF + PPTX + Verbalized — 총 3종)
import os

os.environ.update({
    "AZURE_SEARCH_SERVICE_ENDPOINT": SEARCH_ENDPOINT,
    "AZURE_OPENAI_ENDPOINT": OPENAI_ENDPOINT,
    "AZURE_STORAGE_RESOURCE_ID": STORAGE_RESOURCE_ID,
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT": "text-embedding-3-large",
    "AZURE_STORAGE_CONTAINER_NAME": "raw-documents",
    "SKILLS_FUNCTION_URL": SKILLS_FUNCTION_URL,
    "SKILLS_FUNCTION_KEY": SKILLS_FUNCTION_KEY,
})

from src.pipeline import setup_multimodal_pipeline
setup_multimodal_pipeline(source="st", pipeline="all")

AI Search 멀티모달 파이프라인 설정
  search      : https://search-ragi-63325wdo.search.windows.net
  container   : raw-documents/raw/
  skills func : https://func-skills-ragi-63325wdo.azurewebsites.net
  pipeline    : all  (pdf=True, pptx=True, verbalized=True)
  schedule    : PT24H (start: 2026-01-01T07:00:00Z)
  chunk       : max 2000 chars, overlap 200

[0] Cleanup dependent indexers (so datasource can be recreated)

[1] Data Source
  [datasource] st-raw-datasource
    ✓ created (raw-documents/raw/)

[2A] Pipeline PDF (DI Layout → markdown_split[header+length] → Embedding)
  [index] st-multimodal-pdf-index
    ✓ created
  [skillset] st-multimodal-pdf-skillset  (file_type=pdf)
    ✓ upserted (DI Layout → markdown_split → Embedding)
  [indexer] st-multimodal-pdf-indexer
    ✓ upserted (every PT24H)

[2B] Pipeline PPTX (DI Layout → pptx_page_split[slide] → Embedding)
  [index] st-multimodal-pptx-index
    ✓ created
  [skillset] st-multimodal-pptx-skillset  (file_type=pptx)
    ✓ upserted (DI Layo

In [22]:
# 멀티모달 파이프라인 등록 확인 (3종: pdf / pptx / verbalized)
from azure.identity import DefaultAzureCredential
import requests

credential = DefaultAzureCredential(
    exclude_managed_identity_credential=True,
    exclude_workload_identity_credential=True,
)
token = credential.get_token("https://search.azure.com/.default").token
headers = {"Authorization": f"Bearer {token}"}

PIPELINES = ["pdf", "pptx", "verbalized"]
RESOURCE_TO_SINGULAR = {"indexes": "index", "indexers": "indexer", "skillsets": "skillset"}

for resource, singular in RESOURCE_TO_SINGULAR.items():
    for p in PIPELINES:
        name = f"st-multimodal-{p}-{singular}"
        r = requests.get(f"{SEARCH_ENDPOINT}/{resource}/{name}?api-version=2024-07-01", headers=headers)
        status = "✓" if r.status_code == 200 else f"✗ ({r.status_code})"
        print(f"  {status} {resource}/{name}")


  ✓ indexes/st-multimodal-pdf-index
  ✓ indexes/st-multimodal-pptx-index
  ✓ indexes/st-multimodal-verbalized-index
  ✓ indexers/st-multimodal-pdf-indexer
  ✓ indexers/st-multimodal-pptx-indexer
  ✓ indexers/st-multimodal-verbalized-indexer
  ✓ skillsets/st-multimodal-pdf-skillset
  ✓ skillsets/st-multimodal-pptx-skillset
  ✓ skillsets/st-multimodal-verbalized-skillset


### 5d. Foundry Agent Service Project Endpoint

`04-search-and-query.ipynb` 의 시나리오 E/F (Foundry Agent Service) 가 사용하는 권한·엔드포인트.

| 권한 | Principal | Scope | 용도 |
|---|---|---|---|
| `Cognitive Services OpenAI User` | `userObjectIds` | AIServices 계정 | OpenAI 모델 호출 |
| `Cognitive Services User` | `userObjectIds` | AIServices 계정 | AI Services 키/엔드포인트 조회 |
| `Azure AI User` | `userObjectIds` | AIServices 계정 | Foundry project + agent 생성/조회/실행 |

> ✅ **Bicep 자동 처리 (sweden / sweden-public / korea 공통)** — `infra/{변형}/modules/openai.bicep` 에서 `userObjectIds` 파라미터로 전달된 모든 사용자에게 위 역할이 자동 부여되고, Foundry default project (`proj-ragi-default`) 도 함께 생성됩니다. 이 셀은 배포 output 의 `foundryProjectEndpoint` 를 변수로 읽어옵니다.


In [23]:
# Bicep output 에서 Foundry project endpoint 조회 (sweden / sweden-public / korea 모두 지원)
FOUNDRY_PROJECT_ENDPOINT = outputs.get("foundryProjectEndpoint", {}).get("value", "")
if FOUNDRY_PROJECT_ENDPOINT:
    print(f"Foundry project endpoint: {FOUNDRY_PROJECT_ENDPOINT}")
else:
    print(f"  → 배포 output 에 foundryProjectEndpoint 가 없습니다. Bicep 을 최신 버전으로 재배포 필요.")


Foundry project endpoint: https://ais-ragi-63325wdo.services.ai.azure.com/api/projects/proj-ragi-default


In [24]:
# AI Search Public Access 다시 비활성화 (private 변형에서만 필요)
if IS_PUBLIC:
    print("ℹ️ public 변형: AI Search public access 그대로 유지 — 건너뜀")
else:
    !az search service update \
        --name {SEARCH_NAME} \
        --resource-group {RG_NAME} \
        --public-network-access disabled \
        --output none

    print("AI Search public access 비활성화 완료")


ℹ️ public 변형: AI Search public access 그대로 유지 — 건너뜀


## 6. .env 파일 생성

배포 출력값으로 `.env` 파일을 자동 생성합니다.  
모든 인증은 **Managed Identity (DefaultAzureCredential)** 기반으로, API 키가 필요 없습니다.

In [ ]:
CRAWL_FUNC_URL  = outputs.get('crawlFunctionUrl', {}).get('value', '')
LOGIC_APP_NAME  = outputs.get('crawlLogicAppName', {}).get('value', '')
DI_ENDPOINT     = outputs['docIntelligenceEndpoint']['value']

env_content = f"""# Auto-generated from Bicep deployment — DO NOT commit to git
AZURE_SUBSCRIPTION_ID={SUBSCRIPTION_ID}
AZURE_RESOURCE_GROUP={RG_NAME}
AZURE_LOCATION={LOCATION}
DEPLOY_REGION={DEPLOY_REGION}

# Tenant ID (멀티모달 노트북 인증에 필요)
AZURE_TENANT_ID={subprocess.run(["az", "account", "show", "--query", "tenantId", "-o", "tsv"], capture_output=True, text=True).stdout.strip()}
AZURE_TENANT_ID={_oid_res.stdout.strip() and subprocess.run(["az", "account", "show", "--query", "tenantId", "-o", "tsv"], capture_output=True, text=True).stdout.strip()}

# Storage (Private — Managed Identity 인증)
AZURE_STORAGE_ACCOUNT_NAME={STORAGE_ACCOUNT}
AZURE_STORAGE_CONTAINER_NAME=raw-documents
AZURE_STORAGE_RESOURCE_ID={STORAGE_RESOURCE_ID}
BLOB_CONTAINER_NAME=raw-documents

# Azure AI Services (OpenAI)
AZURE_OPENAI_ENDPOINT={OPENAI_ENDPOINT}
AZURE_OPENAI_EMBEDDING_DEPLOYMENT=text-embedding-3-large
AZURE_OPENAI_GPT54_DEPLOYMENT={GPT_DEPLOY_NAME}

# Azure AI Search
AZURE_SEARCH_SERVICE_ENDPOINT={SEARCH_ENDPOINT}
AZURE_SEARCH_INDEX_NAME=law-documents-index

# Document Intelligence
AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT={DI_ENDPOINT}

# Crawl Pipeline
AZURE_FUNCTION_CRAWL_URL={CRAWL_FUNC_URL}
AZURE_FUNCTION_CRAWL_ORCH_URL={CRAWL_FUNC_URL.rsplit('/api/', 1)[0]}/api/orchestrators/crawl_preprocess
AZURE_LOGIC_APP_NAME={LOGIC_APP_NAME}

# Skills Function (Multimodal Pipeline)
SKILLS_FUNCTION_URL={SKILLS_FUNCTION_URL}
SKILLS_FUNCTION_KEY={SKILLS_FUNCTION_KEY}

# Foundry Agent Service (Sweden 전용 — Knowledge Agent + agent demo)
FOUNDRY_PROJECT_ENDPOINT={FOUNDRY_PROJECT_ENDPOINT}
KNOWLEDGE_AGENT_NAME=legal-knowledge-agent
"""

with open('../.env', 'w') as f:
    f.write(env_content)

print(".env 파일 생성 완료 (Managed Identity 기준 변수 포함)")
print(env_content)


## 7. 배포 검증

In [26]:
# 리소스 그룹 내 전체 리소스 확인
!az resource list --resource-group {RG_NAME} --output table

# Korea: DI 별도 RG 리소스도 확인
if cfg["has_di_rg"]:
    print(f"\n=== DI RG ({DI_RG}) ===")
    !az resource list --resource-group {DI_RG} --output table

Name                                 ResourceGroup                Location       Type                                           Status
-----------------------------------  ---------------------------  -------------  ---------------------------------------------  ---------
stragi63325wdoby                     rg-rag-indexing-lab-swc-pub  swedencentral  Microsoft.Storage/storageAccounts              Succeeded
nsg-pep-ragi-63325wdo                rg-rag-indexing-lab-swc-pub  swedencentral  Microsoft.Network/networkSecurityGroups        Succeeded
di-ragi-63325wdo                     rg-rag-indexing-lab-swc-pub  swedencentral  Microsoft.CognitiveServices/accounts           Succeeded
ais-ragi-63325wdo                    rg-rag-indexing-lab-swc-pub  swedencentral  Microsoft.CognitiveServices/accounts           Succeeded
vnet-ragi-63325wdo                   rg-rag-indexing-lab-swc-pub  swedencentral  Microsoft.Network/virtualNetworks              Succeeded
asp-crawl-cons-ragi-63325wdo         

## 8. 초기 데이터 시드 로드

`pubzip0513143342/raw-documents-seed` 에 미리 패키징된 블롭(185K 파일, 1.45 GiB)을 **azcopy server-to-server copy** 로 새 스토리지 계정의 `raw-documents` 컨테이너에 복사합니다.

- 호출 사용자는 `pubzip0513143342` 스토리지에 **Storage Blob Data Reader** 이상의 RBAC 권한이 필요합니다.
- 대상 스토리지에는 **Storage Blob Data Contributor** 역할이 필요합니다.
- 이미 직접 크롤링을 수행했다면 이 셀은 건너뛰셔도 됩니다.

In [27]:
# ── 시드 데이터 업로드 (pubzip → raw-documents) ──
# pubzip0513143342/raw-documents-seed 에 미리 패키징된 블롭을 azcopy 로 복사
import subprocess, os

DEST_ACCOUNT = outputs.get('storageAccountName', {}).get('value') or os.environ.get('AZURE_STORAGE_ACCOUNT_NAME')
DEST_CONTAINER = 'raw-documents'
SRC_ACCOUNT = 'pubzip0513143342'
SRC_CONTAINER = 'raw-documents-seed'

print(f"Seeding {DEST_ACCOUNT}/{DEST_CONTAINER} from {SRC_ACCOUNT}/{SRC_CONTAINER} via azcopy ...")

env = {**os.environ, "AZCOPY_AUTO_LOGIN_TYPE": "AZCLI"}
result = subprocess.run(
    ["/usr/bin/azcopy", "copy",
     f"https://{SRC_ACCOUNT}.blob.core.windows.net/{SRC_CONTAINER}/*",
     f"https://{DEST_ACCOUNT}.blob.core.windows.net/{DEST_CONTAINER}/",
     "--recursive=true", "--overwrite=ifSourceNewer",
     "--s2s-preserve-access-tier=false"],
    capture_output=True, text=True, env=env, timeout=3600,
)
# azcopy 요약만 출력
lines = result.stdout.splitlines()
summary_start = next((i for i, l in enumerate(lines) if 'summary' in l.lower()), max(0, len(lines)-20))
print("\n".join(lines[summary_start:]))
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

Seeding stragi63325wdoby/raw-documents from pubzip0513143342/raw-documents-seed via azcopy ...
INFO: Scanning...
INFO: Authenticating to destination using Azure AD
INFO: Authenticating to source using Azure AD
INFO: Any empty folders will not be processed, because source and/or destination doesn't have full folder support

failed to perform copy command due to error: cannot start job due to error cannot list files due to reason GET https://pubzip0513143342.blob.core.windows.net/raw-documents-seed
--------------------------------------------------------------------------------
RESPONSE 403: 403 This request is not authorized to perform this operation.
ERROR CODE: AuthorizationFailure
--------------------------------------------------------------------------------
﻿<?xml version="1.0" encoding="utf-8"?><Error><Code>AuthorizationFailure</Code><Message>This request is not authorized to perform this operation.
RequestId:f5eb5332-d01e-004c-51a6-e9ebc9000000
Time:2026-05-22T04:48:19.7925623Z<

In [28]:
# import subprocess, json, random

# ACCOUNT = STORAGE_ACCOUNT
# CONTAINER = "raw-documents"
# SAMPLE_PER_DIR = 5  # 카테고리(HA/SS/ST)별 유지할 파일 수

# def _list_blobs(prefix: str, ext: str) -> list[str]:
#     """prefix 아래 ext 확장자 blob 목록 반환."""
#     r = subprocess.run(
#         ["az", "storage", "blob", "list",
#          "--account-name", ACCOUNT, "--container-name", CONTAINER,
#          "--prefix", prefix, "--auth-mode", "login",
#          "--query", f"[?ends_with(name, '.{ext}')].name",
#          "-o", "tsv"],
#         capture_output=True, text=True
#     )
#     return [b for b in r.stdout.strip().split("\n") if b]

# def _delete_blobs(names: list[str]) -> int:
#     """blob 목록 일괄 삭제 (batch 100개씩)."""
#     deleted = 0
#     for i in range(0, len(names), 100):
#         batch = names[i:i+100]
#         # az storage blob delete-batch 는 패턴만 지원하므로 개별 삭제
#         for name in batch:
#             subprocess.run(
#                 ["az", "storage", "blob", "delete",
#                  "--account-name", ACCOUNT, "--container-name", CONTAINER,
#                  "--name", name, "--auth-mode", "login", "-o", "none"],
#                 capture_output=True, text=True
#             )
#             deleted += 1
#     return deleted

# # ── 카테고리별 샘플링 + 나머지 삭제 ──
# DIRS = ["HA", "SS", "ST"]
# total_keep, total_delete = 0, 0

# for ext, prefix in [("pdf", "raw/pdf"), ("pptx", "raw/pptx")]:
#     print(f"\n=== {ext.upper()} 샘플링 (카테고리별 {SAMPLE_PER_DIR}개) ===")
#     for d in DIRS:
#         blobs = _list_blobs(f"{prefix}/{d}/", ext)
#         if len(blobs) <= SAMPLE_PER_DIR:
#             print(f"  {d}: {len(blobs)}개 (전체 유지)")
#             total_keep += len(blobs)
#             continue
#         random.seed(42)  # 재현 가능한 샘플링
#         keep = set(random.sample(blobs, SAMPLE_PER_DIR))
#         to_delete = [b for b in blobs if b not in keep]
#         print(f"  {d}: {len(blobs)}개 → {len(keep)}개 유지, {len(to_delete)}개 삭제 중...")
#         deleted = _delete_blobs(to_delete)
#         total_keep += len(keep)
#         total_delete += deleted
#         print(f"    ✓ {deleted}개 삭제 완료")

# print(f"\n{'='*50}")
# print(f"결과: 유지 {total_keep}개 / 삭제 {total_delete}개")
# print(f"멀티모달 인덱싱 테스트용 ~30개 파일만 남김")

---
다음 단계: [02-data-crawling.ipynb](02-data-crawling.ipynb) — Azure Function App으로 크롤링 실행 및 모니터링